# Testează-te — Grading Test Notebook
Vasile Bria | RoBacTutor

Tests whether the existing adapter can score a student's answer against the
official barem and explain what's missing, using the real model, tokenizer
settings, and chat template from `RoBacTutor_FineTuning_FINAL_2.ipynb`.

**Run cells in order.** Cell 6 prints a sample response — check the points
notation there before trusting `points_total`, and check the real subject
labels before setting `TEST_SUBJECT` in the last cell.

## Cell 1 — Install packages

NOTE: using current versions here instead of pinning to the exact
fine-tuning notebook versions (transformers==4.46.0). That older pinned
version hits a known transformers bug (`_validate_bnb_multi_backend_availability`
tries to call `.discard()` on a frozenset) that was fixed in a later
release. Since this is inference/loading, not reproducing a training run,
exact version parity isn't needed here -- just working versions.

In [ ]:
!pip install -q -U \
    transformers \
    bitsandbytes \
    peft \
    accelerate \
    sentencepiece

import os
os.kill(os.getpid(), 9)  # restart runtime after install

## Cell 2 — Upload your trained adapter zip
Run this cell **after the runtime restarts** from Cell 1.

In [ ]:
from google.colab import files
import zipfile

print("Upload robactutor_lora_adapters_BRI23222497.zip (or your current adapter zip)...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

ADAPTER_PATH = "./robactutor-lora-adapters"
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall(ADAPTER_PATH)
print(f"Adapter extracted to {ADAPTER_PATH}")

## Cell 3 — Upload the question dataset (157 or 166-pair version)

In [ ]:
print("Upload robactutor_sft_dataset_expanded.jsonl (or whichever version you're testing)...")
uploaded = files.upload()
DATASET_FILE = list(uploaded.keys())[0]

## Cell 4 — Load base model + adapter
Matches Cell 5 of your fine-tuning notebook exactly (same model ID, same 4-bit config, same tokenizer settings).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "OpenLLM-Ro/RoMistral-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {MODEL_ID} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("✅ Base model + adapter loaded")

## Cell 5 — Generation function
Same base sampling params as your Cell 9 qualitative eval (top_p 0.9, repetition_penalty 1.1). Temperature is now a parameter — 0.7 is fine for open-ended explanation, but a precise, format-locked task like grading needs much lower temperature to stay on task.

In [ ]:
def call_adapter_model(system_prompt: str, instruction: str, max_new_tokens: int = 400, temperature: float = 0.7) -> str:
    prompt = f"<s>[INST] {system_prompt}\n\n{instruction} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0.01,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## Cell 6 — Load question bank
Uses the real schema: `system` / `instruction` / `response` / `metadata.subject`.

**Check the printed sample below** — confirm the points notation matches the regex, and note the real subject label strings for later.

In [ ]:
import json
import re
import random
from collections import Counter
from typing import Optional


def extract_total_points(response_text: str) -> int:
    # Primary: an explicit declared total near the start, e.g. "(6 puncte)"
    match = re.search(r"\((\d+)\s*puncte\)", response_text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))

    # Fallback: "N puncte" without parentheses, anywhere in the text
    match = re.search(r"(\d+)\s*puncte", response_text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))

    # Last resort: sum standalone "(Np)" / "(Np.)" markers -- use with caution,
    # some barems use these as per-tier partial-credit options (e.g. "1 p.X3")
    # rather than additive components, so this can overcount.
    matches = re.findall(r"\((\d+)\s*p\.?\)", response_text, flags=re.IGNORECASE)
    return sum(int(m) for m in matches) if matches else 0


def load_question_bank(jsonl_path: str) -> list[dict]:
    bank = []
    with open(jsonl_path, encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            bank.append({
                "id": f"q_{i}",
                "subject": row["metadata"]["subject"],
                "instruction": row["instruction"],
                "barem": row["response"],
                "points_total": extract_total_points(row["response"]),
            })
    return bank


question_bank = load_question_bank(DATASET_FILE)
print(f"Loaded {len(question_bank)} questions")
print("Subject distribution:", Counter(q["subject"] for q in question_bank))

sample = question_bank[0]
print("\nSample response text (check points format here):")
print(sample["barem"][:300])
print(f"Detected points_total: {sample['points_total']}  <- should be > 0. If it's 0, fix the regex above.")

## Cell 7 — Auto-pick logic
Student picks the subject, bot auto-picks an item within it, no repeats within a session.

In [ ]:
class SessionPicker:
    def __init__(self, bank: list[dict]):
        self.bank = bank
        self.asked_ids: set[str] = set()

    def pick(self, subject: str) -> Optional[dict]:
        pool = [q for q in self.bank if q["subject"] == subject and q["id"] not in self.asked_ids]
        if not pool:
            pool = [q for q in self.bank if q["subject"] == subject]
            if not pool:
                return None
        chosen = random.choice(pool)
        self.asked_ids.add(chosen["id"])
        return chosen

## Cell 8 — Grading prompt
Added a one-shot worked example this time — zero-shot instruction-following against a dominant trained habit (the barem-format reflex) is often unreliable; seeing one concrete example of the expected shape tends to help a lot.

In [ ]:
GRADING_SYSTEM_PROMPT = """Ênești un profesor examinator care corectează lucrări de Bacalaureat conform
baremului oficial de evaluare.

Vei primi trei elemente:
1. CERINȚA — întrebarea sau sarcina din subiectul de examen
2. BAREMUL OFICIAL — răspunsul corect, punctat pe criterii
3. RĂSPUNSUL ELEVULUI — ce a scris elevul

Sarcina ta:
1. Acordă un punctaj din totalul de puncte disponibil în barem, evaluând
   fiecare criteriu de punctare separat.
2. Dacă răspunsul e parțial sau greșit, explică EXACT ce a lipsit sau ce
   este incorect, raportat la barem.
3. Dacă răspunsul e corect, confirmă și explică pe scurt de ce îndeplinește
   baremul.
4. Nu inventa criterii care nu sunt în barem. NU genera un alt barem sau altă
   cerință — folosește DOAR ce primești mai jos.
5. Răspunde STRICT în formatul JSON, fără text înainte sau după.

EXEMPLU:
CERINȚA: Explică rolul Curentului Golfului în clima Europei.
BAREMUL OFICIAL (punctaj total: 3): Curentul Golfului transportă apă caldă
dinspre Golful Mexic spre Europa de Vest (1p), încălzind clima regiunii (1p) și
explicând de ce Europa de Vest are ierni mai blânde decât alte zone de la aceeași
latitudine (1p).
RĂSPUNSUL ELEVULUI: apa

RĂSPUNS AȘTEPTAT:
{
  "punctaj_acordat": 0,
  "punctaj_total": 3,
  "criterii_indeplinite": [],
  "criterii_lipsa": ["originea apei calde din Golful Mexic", "efectul de încălzire asupra climei", "explicația pentru iernile blânde"],
  "explicatie": "Răspunsul 'apa' nu menționează niciunul dintre cele trei criterii din barem și nu constituie un răspuns la întrebare."
}

Acum notează următorul răspuns real, folosind DOAR cerința și baremul de mai jos:"""


def build_grading_instruction(question: dict, student_answer: str) -> str:
    return f"""MATERIE: {question['subject']}

CERINȚA:
{question['instruction']}

BAREMUL OFICIAL (punctaj total: {question['points_total']}):
{question['barem']}

RĂSPUNSUL ELEVULUI:
{student_answer}"""

## Cell 9 — Interactive: real question, your own answer, graded against the barem
Picks a random real question from the subject you choose, lets you type an answer, then grades it. Run this cell again any time to get a new question.

In [ ]:
picker = SessionPicker(question_bank)

TEST_SUBJECT = input("Alege materia (limba_romana / limba_engleza / istorie / matematica): ").strip()
question = picker.pick(TEST_SUBJECT)

if question is None:
    print(f"Nu exista intrebari pentru materia '{TEST_SUBJECT}' -- verifica etichetele reale afisate mai sus")
else:
    print("\n--- CERINTA ---")
    print(question["instruction"])
    print(f"\n(Punctaj total: {question['points_total']})")

    student_answer = input("\nScrie raspunsul tau:\n")

    grading_instruction = build_grading_instruction(question, student_answer)
    raw_output = call_adapter_model(GRADING_SYSTEM_PROMPT, grading_instruction, temperature=0.2)
    print("\nRAW MODEL OUTPUT (WITH ADAPTER):\n", raw_output)

    try:
        result = json.loads(raw_output)
        print("\nPARSED:\n", json.dumps(result, indent=2, ensure_ascii=False))
    except json.JSONDecodeError:
        print("\n⚠️ Not valid JSON -- inspect raw output above (model may need stricter prompting or lower temperature)")

## Cell 10 — Same test, base model only (adapter bypassed)
Key diagnostic: the barem-habit lives in the ADAPTER's weights specifically. The unmodified base model was never trained on that reflex. If grading works here but not in Cell 9, that means: keep the adapter for generating barem answers (what it's good at), but route grading through the base model instead -- an architecture fix, not a training problem.

Run this right after Cell 9 so you're testing the SAME question and answer for a fair comparison.

IMPORTANT: use `model.disable_adapter()`, not `base_model.generate()` directly -- PEFT modifies the base model's layers IN PLACE, so `base_model` and the adapter-attached `model` are the same underlying weights unless the adapter is explicitly disabled via this context manager.

In [ ]:
# Reuses `question`, `student_answer`, and `grading_instruction` from Cell 9 -- run Cell 9 first

with model.disable_adapter():
    prompt = f"<s>[INST] {GRADING_SYSTEM_PROMPT}\n\n{grading_instruction} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=400,
            temperature=0.2,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    raw_output_base = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

print("RAW MODEL OUTPUT (ADAPTER GENUINELY DISABLED):\n", raw_output_base)

try:
    result_base = json.loads(raw_output_base)
    print("\nPARSED:\n", json.dumps(result_base, indent=2, ensure_ascii=False))
except json.JSONDecodeError:
    print("\n⚠️ Not valid JSON either -- inspect raw output above")

## Cell 11 — Hybrid grading: semantic similarity (no LLM needed for scoring or feedback)

Both the adapter and the base model reliably substitute a generic, memorized
barem template ("amplasarea in timp... doua date cronologice") regardless of
the real question or barem given in-context -- confirmed across adapter-on,
adapter-off, low temperature, and a worked example. That's a real limitation
of RoMistral-7B-Instruct's parametric knowledge overriding in-context
instructions for this task shape, not a bug to keep chasing with prompts.

This cell sidesteps the problem entirely: score by semantic similarity
between the student's answer and the REAL official barem (deterministic, no
hallucination possible), and show the real barem text as feedback instead of
asking a model to invent an explanation.

In [ ]:
!pip install -q -U sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

embed_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")


def grade_answer_hybrid(question: dict, student_answer: str) -> dict:
    emb_student = embed_model.encode(student_answer, convert_to_tensor=True)
    emb_barem = embed_model.encode(question["barem"], convert_to_tensor=True)
    similarity = util.cos_sim(emb_student, emb_barem).item()

    # NOTE: this similarity -> points mapping is a rough starting calibration.
    # Once you have a handful of real graded examples, check whether known-good
    # answers score meaningfully higher than known-wrong ones and adjust the
    # scaling (or add a minimum-similarity threshold below which score = 0).
    similarity_clamped = max(0.0, similarity)
    estimated_points = round(similarity_clamped * question["points_total"], 1)

    return {
        "punctaj_estimat": estimated_points,
        "punctaj_total": question["points_total"],
        "similaritate": round(similarity_clamped, 3),
        "barem_oficial": question["barem"],
    }


# Test on the same question/answer already in memory from Cell 9
result_hybrid = grade_answer_hybrid(question, student_answer)
print(f"Punctaj estimat: {result_hybrid['punctaj_estimat']} / {result_hybrid['punctaj_total']}"
      f"  (similaritate: {result_hybrid['similaritate']})")
print("\nBarem oficial (pentru comparatie):")
print(result_hybrid["barem_oficial"])